# Overall confusion matrices on the test set (mask models)

One confusion matrix per model, combining all of its runs on the 290 test nodules:

1. Single split, model A: seeds 1-3 combined (`runs/cls_no_fold_mask`)
2. Single split, model B: seeds 1-3 combined
3. Folds, model A: the 4 fold models combined (`runs/cls_mask`)
4. Folds, model B: the 4 fold models combined
5. Folds ensemble, models A and B: already made by `score_test_fold.py`, shown at the end

"Combined" means every run's predictions are counted together, so each test nodule is counted once per run:
3 × 290 = 870 predictions for single split, 4 × 290 = 1,160 for folds.
Drawn with `_confusion_png` from `plots.py`, the same function as the per-run matrices:
cell numbers are counts, colour is the share of the row.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
from IPython.display import Image, display

PROJECT = Path.cwd().parent if Path.cwd().name == "Model" else Path.cwd()
sys.path.insert(0, str(PROJECT / "Model"))       # plots.py
from plots import _confusion_png

SINGLE_DIR = PROJECT / "runs" / "cls_no_fold_mask"
FOLDS_DIR = PROJECT / "runs" / "cls_mask"
SINGLE_OUT = PROJECT / "results" / "test_no_fold_mask"     # saved next to the other test results
FOLDS_OUT = PROJECT / "results" / "test_fold_mask"
SEEDS = [1, 2, 3]
FOLDS = [1, 2, 3, 4]
P_COLS = [f"p{k}" for k in range(1, 6)]

labels = pd.read_csv(PROJECT / "data" / "nodules_labels.csv")
test = labels[labels["split"] == "test"].set_index("nodule_key")
hard = test["hard"].to_numpy()
print(f"{len(test)} test nodules")

## Combine each model's runs

In [ ]:
def predicted(run_dir):
    """One run's predicted rating for each test nodule (the rating with the highest probability)."""
    preds = pd.read_csv(run_dir / "test_preds.csv").set_index("nodule_key")
    return preds.loc[test.index, P_COLS].to_numpy().argmax(axis=1) + 1

matrices = []     # (title, true ratings, predicted ratings, output file)
for kind, model in (("hard", "A"), ("soft", "B")):
    pred = np.concatenate([predicted(SINGLE_DIR / f"{kind}_seed{s}") for s in SEEDS])
    matrices.append((f"Single split, model {model}\nseeds 1-3 combined, {len(pred)} predictions",
                     np.tile(hard, len(SEEDS)), pred, SINGLE_OUT / f"confusion_{kind}_pooled.png"))
for kind, model in (("hard", "A"), ("soft", "B")):
    pred = np.concatenate([predicted(FOLDS_DIR / f"{kind}_fold{k}") for k in FOLDS])
    matrices.append((f"Folds, model {model}\n4 fold models combined, {len(pred)} predictions",
                     np.tile(hard, len(FOLDS)), pred, FOLDS_OUT / f"confusion_{kind}_folds_pooled.png"))

## Draw and show the four combined matrices

In [ ]:
matplotlib.use("Agg")      # _confusion_png saves to a file; the saved file is shown below
for title, h, pred, path in matrices:
    _confusion_png(h, pred, title, path)
    print(path.relative_to(PROJECT))
    display(Image(filename=str(path), width=460))

## The ensembles

The ensemble averages the 4 fold models' probabilities before choosing a rating, so its predictions are not the
same as the 4 fold models' predictions counted together (above). For example, the model B ensemble predicts 4 for
none of the 44 nodules rated 4, while its fold models get 3.25 of them right on average.

`score_test_fold.py` already saves `confusion_hard_ensemble.png` and `confusion_soft_ensemble.png`, but their
"median reader rating" axis label is cut off. These are remade here with two-line titles so the label fits,
under new names (`..._ensemble_report.png`), leaving the scoring script's files unchanged.

In [ ]:
def probs_of(run_dir):
    """One run's test probabilities, rows in the order of the test nodules."""
    preds = pd.read_csv(run_dir / "test_preds.csv").set_index("nodule_key")
    return preds.loc[test.index, P_COLS].to_numpy()

for kind, model in (("hard", "A"), ("soft", "B")):
    # the ensemble, as in score_test_fold.py: average the 4 fold models' probabilities per nodule
    ensemble = np.mean([probs_of(FOLDS_DIR / f"{kind}_fold{k}") for k in FOLDS], axis=0)
    path = FOLDS_OUT / f"confusion_{kind}_ensemble_report.png"
    _confusion_png(hard, ensemble.argmax(axis=1) + 1,
                   f"Folds ensemble, model {model}\n4 fold models averaged, {len(hard)} nodules", path)
    print(path.relative_to(PROJECT))
    display(Image(filename=str(path), width=460))
